# Complementary (Grounded) Concentric Transmon
## Full Design, Simulation and Quantum Analysis — Qiskit-Metal + pyEPR

### Geometry (v2 — explicit metallic plane)

```
        ┌─────────────────────────────────┐
        │           W x W                 │
        │       metallic plane            │
        │                                 │
        │        ╭──────────╮  ←Wslot     │
        │      ╭─╯  ○ disk  ╰─╮           │
        │  ──JJ─  Rdisk   JJ──            │
        │      ╰─╮          ╭─╯           │
        │        ╰──────────╯             │
        └─────────────────────────────────┘
```

| Parameter | Description | Design knob |
|---|---|---|
| `W` | Side of outer metallic square | Isolation / boundary condition |
| `Rdisk` | Radius of central metallic disk | Primary Ec knob (↑Rdisk → ↓Ec) |
| `Wslot` | Width of circular slot (gap) | Secondary Ec knob (↑Wslot → ↓Ec) |
| `jj_width` | Tangential width of each JJ bridge | JJ area / Ic spread |
| `Lj` | Josephson inductance (HFSS variable) | Ej → f_01 |

### Design targets
| Parameter | Target |
|---|---|
| f_01 | 3 – 4.8 GHz |
| Ec | ~140 MHz |
| Ej/Ec | > 50 |

### Reference: rfphotoniqlabs/qiskit-metal
- `tutorials/4 Analysis/A. Core/4.02 Eigenmode and EPR.ipynb`


In [1]:
%reload_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

import qiskit_metal as metal
from qiskit_metal import designs, draw
from qiskit_metal import MetalGUI, Dict
from qiskit_metal.toolbox_python.attr_dict import Dict
from qiskit_metal.qlibrary.core import QComponent

print(f"Qiskit-Metal version: {metal.__version__}")


Qiskit-Metal version: 0.5.3.post1


---
## Step 1 — Custom QComponent: `GroundedConcentricTransmon`

### Geometry construction (inside `make()`)

1. **`metal_plane`** — W×W rectangle with a circular hole of radius
   `Rdisk + Wslot` removed via shapely `.difference()`.
   This is the explicit local ground plane of the qubit.

2. **`disk`** — filled circle of radius `Rdisk`.
   The metallic island (qubit capacitor pad).

3. **`jj_east` / `jj_west`** — thin rectangles of width `jj_width` and
   height `Wslot`, centred at radial distance `Rdisk + Wslot/2` on the
   ±x axis. They bridge the slot exactly.

All three are assembled in the component's local frame, then rotated and
translated to (`pos_x`, `pos_y`) with orientation `orientation`.

### Why a shapely `.difference()` for the slot?
Rather than registering the slot as a separate `subtract=True` geometry
(which removes from the chip-level ground plane), we pre-cut the W×W pad
inside `make()` using shapely. This makes the component fully self-contained —
the hole is baked into the `metal_plane` polygon before it reaches HFSS.


In [2]:
from qiskit_metal import draw, Dict
from qiskit_metal.toolbox_python.attr_dict import Dict
from qiskit_metal.qlibrary.core import QComponent

'''
class GroundedConcentricTransmon(QComponent):
    """Complementary (grounded) concentric transmon qubit.

    Layout
    ------
    A W x W metallic plane (the local ground) has a circular slot of
    width Wslot cut from its centre.  A metallic disk of radius Rdisk
    sits inside the slot.  Two Josephson Junctions bridge the slot
    radially at east (+x) and west (-x).

    Parameters (all length strings, e.g. '200um')
    -----------------------------------------------
    W        : side length of the outer metallic square plane
    Rdisk    : radius of the central metallic disk (island)
    Wslot    : radial width of the circular slot (= capacitive gap)
    jj_width : tangential width of each JJ bridge rectangle

    Notes
    -----
    The circular hole in metal_plane has radius = Rdisk + Wslot.
    The JJs span exactly the slot (height = Wslot) at +/-x.
    Both JJs share the same HFSS inductance variable Lj.
    Effective parallel inductance: Lj_eff = Lj/2.
    Effective Josephson energy:    Ej_eff = 2 * (Phi0/2pi)^2 / Lj.
    """

    component_metadata = Dict(
        short_name='GCT',
        _qgeometry_table_poly='True',
        _qgeometry_table_junction='True',
    )

    default_options = Dict(
        pos_x='0um',
        pos_y='0um',
        orientation='0',
        # Geometry
        W='800um',           # side of the outer metallic square plane
        Rdisk='200um',       # radius of the central metallic disk
        Wslot='30um',        # width of the circular slot (capacitive gap)
        jj_width='10um',     # tangential width of each JJ bridge
        # Layer / chip
        chip='main',
        layer='1',
        # HFSS variables
        hfss_inductance='Lj',
        hfss_capacitance='Cj',
        hfss_resistance='',
        hfss_mesh_kw_jj='7um',
    )

    def make(self):
        p = self.p

        # ── 1. W x W metallic plane with circular slot removed ────────────
        # The slot is a ring from Rdisk to Rdisk+Wslot; we remove the whole
        # disc of radius Rdisk+Wslot to create the hole, leaving the disk
        # island region empty (it will be filled by the disk polygon below).
        slot_radius   = p.Rdisk + p.Wslot           # outer radius of slot
        metal_plane   = draw.rectangle(p.W, p.W)    # full W x W pad
        circular_hole = draw.Point(0, 0).buffer(slot_radius)
        metal_plane   = metal_plane.difference(circular_hole)  # cut the hole

        # ── 2. Central metallic disk (qubit island) ───────────────────────
        disk = draw.Point(0, 0).buffer(p.Rdisk)

        # ── 3. JJ rectangles — span the slot radially ─────────────────────
        # Each JJ is a thin rectangle:
        #   - height = Wslot (spans the full slot width)
        #   - width  = jj_width (tangential footprint)
        #   - centred at radial midpoint of the slot
        jj_center_r = p.Rdisk + p.Wslot / 2
        jj_h        = p.Wslot

        jj_east = draw.rectangle(p.jj_width, jj_h,  jj_center_r, 0)
        jj_west = draw.rectangle(p.jj_width, jj_h, -jj_center_r, 0)

        # ── 4. Apply orientation and translation ──────────────────────────
        all_geom = [metal_plane, disk, jj_east, jj_west]
        all_geom = draw.rotate(all_geom, p.orientation, origin=(0, 0))
        all_geom = draw.translate(all_geom, p.pos_x, p.pos_y)
        metal_plane, disk, jj_east, jj_west = all_geom

        # ── 5. Register geometries ────────────────────────────────────────
        # Outer metallic plane (W x W minus circular hole)
        self.add_qgeometry('poly', {'metal_plane': metal_plane},
                           layer=p.layer, chip=p.chip)

        # Central disk (qubit island)
        self.add_qgeometry('poly', {'disk': disk},
                           layer=p.layer, chip=p.chip)

        # Josephson Junctions (junction table for HFSS lumped elements)
        jj_opts = dict(
            width=p.jj_width,
            hfss_inductance=p.hfss_inductance,
            hfss_capacitance=p.hfss_capacitance,
            hfss_resistance=p.hfss_resistance,
            hfss_mesh_kw_jj=p.hfss_mesh_kw_jj,
            chip=p.chip,
        )
        self.add_qgeometry('junction', {'jj_east': jj_east}, **jj_opts)
        self.add_qgeometry('junction', {'jj_west': jj_west}, **jj_opts)


print("GroundedConcentricTransmon class defined.")
print()
print("Geometry summary:")
print("  metal_plane  : W x W rectangle with circular hole (radius = Rdisk + Wslot)")
print("  disk         : filled circle (radius = Rdisk)")
print("  jj_east/west : rectangle spanning the slot at +/-x")
'''


'\nclass GroundedConcentricTransmon(QComponent):\n    """Complementary (grounded) concentric transmon qubit.\n\n    Layout\n    ------\n    A W x W metallic plane (the local ground) has a circular slot of\n    width Wslot cut from its centre.  A metallic disk of radius Rdisk\n    sits inside the slot.  Two Josephson Junctions bridge the slot\n    radially at east (+x) and west (-x).\n\n    Parameters (all length strings, e.g. \'200um\')\n    -----------------------------------------------\n    W        : side length of the outer metallic square plane\n    Rdisk    : radius of the central metallic disk (island)\n    Wslot    : radial width of the circular slot (= capacitive gap)\n    jj_width : tangential width of each JJ bridge rectangle\n\n    Notes\n    -----\n    The circular hole in metal_plane has radius = Rdisk + Wslot.\n    The JJs span exactly the slot (height = Wslot) at +/-x.\n    Both JJs share the same HFSS inductance variable Lj.\n    Effective parallel inductance: Lj_eff 

---
## Step 2 — Build the Design and Display in MetalGUI


In [3]:
# ── Create design ─────────────────────────────────────────────────────────
# Set chip size to match W (the metallic plane side).
# A small border is added so the HFSS airbox does not clip the component.
W_um = 800    # must match the W option below

design = designs.DesignPlanar({}, True)
design.chips.main.size['size_x'] = f'{W_um}um'
design.chips.main.size['size_y'] = f'{W_um}um'
design.chips.main.size['center_x'] = '0um'
design.chips.main.size['center_y'] = '0um'

gui = MetalGUI(design)
print("Design created. MetalGUI opened.")


Design created. MetalGUI opened.


In [4]:
from qiskit_metal.qlibrary.user_components.circmong_singleJJ import CircmonG

circmon1 = CircmonG(design, 'TC1', options=dict(
    pos_x      = '0mm',
    pos_y      = '0mm',
    pad_r      = '200um',   # larger disk → lower Ec
    pad2pk_gap = '100um',   # slot width
    jj_width   = '20um',
    jj_angle   = 0,        # JJ placed at 45 deg (NE direction)
    jj_in_pad  = '2um',
    jj_in_pk   = '2um',
))


In [5]:
# ── Instantiate the component ─────────────────────────────────────────────
# Starting values chosen to target Ec ~ 140 MHz, f_01 ~ 3-4.8 GHz.
# Adjust W, Rdisk, Wslot to sweep Ec; adjust Lj (HFSS variable) for f_01.

design.overwrite_enabled = True

'''
cct = GroundedConcentricTransmon(design, 'CCT1', options=Dict(
    pos_x='0um',
    pos_y='0um',
    orientation='0',
    W='800um',       # outer metallic plane size
    Rdisk='200um',   # disk radius  — primary Ec knob
    Wslot='30um',    # slot width   — secondary Ec knob
    jj_width='10um',
    chip='main',
    layer='1',
    hfss_inductance='Lj',
    hfss_capacitance='Cj',
))
'''

gui.rebuild()
gui.autoscale()

'''
print("Component built.")
print()
print("Geometry dimensions:")
print(f"  W         = {cct.options.W}")
print(f"  Rdisk     = {cct.options.Rdisk}")
print(f"  Wslot     = {cct.options.Wslot}")
print(f"  slot hole radius (Rdisk+Wslot) = 230 um")
print(f"  jj_width  = {cct.options.jj_width}")
print()
print("Junction table:")
print(design.qgeometry.tables['junction'][
    ['component','name','width','hfss_inductance','hfss_capacitance']])
'''


'\nprint("Component built.")\nprint()\nprint("Geometry dimensions:")\nprint(f"  W         = {cct.options.W}")\nprint(f"  Rdisk     = {cct.options.Rdisk}")\nprint(f"  Wslot     = {cct.options.Wslot}")\nprint(f"  slot hole radius (Rdisk+Wslot) = 230 um")\nprint(f"  jj_width  = {cct.options.jj_width}")\nprint()\nprint("Junction table:")\nprint(design.qgeometry.tables[\'junction\'][\n    [\'component\',\'name\',\'width\',\'hfss_inductance\',\'hfss_capacitance\']])\n'

In [6]:
'''
# ── Verify geometry using matplotlib renderer ─────────────────────────────
from qiskit_metal.renderers.renderer_mpl.mpl_renderer import QMplRenderer

fig, ax = plt.subplots(1, 1, figsize=(6, 6))
renderer_mpl = design.renderers.mpl
renderer_mpl.render(ax=ax)
ax.set_title('Grounded Concentric Transmon — Metal Layout', fontsize=12)
ax.set_xlabel('x (mm)')
ax.set_ylabel('y (mm)')
plt.tight_layout()
plt.savefig('layout_view.png', dpi=150, bbox_inches='tight')
plt.show()
print("Layout saved: layout_view.png")
'''


'\n# ── Verify geometry using matplotlib renderer ─────────────────────────────\nfrom qiskit_metal.renderers.renderer_mpl.mpl_renderer import QMplRenderer\n\nfig, ax = plt.subplots(1, 1, figsize=(6, 6))\nrenderer_mpl = design.renderers.mpl\nrenderer_mpl.render(ax=ax)\nax.set_title(\'Grounded Concentric Transmon — Metal Layout\', fontsize=12)\nax.set_xlabel(\'x (mm)\')\nax.set_ylabel(\'y (mm)\')\nplt.tight_layout()\nplt.savefig(\'layout_view.png\', dpi=150, bbox_inches=\'tight\')\nplt.show()\nprint("Layout saved: layout_view.png")\n'

In [7]:
# ── Analytical preview: f_01 and Ej/Ec vs Lj (fixed Ec = 140 MHz) ────────
h_planck = 6.626e-34
Phi0     = 2.068e-15

def Lj_to_Ej_GHz(Lj_nH):
    """Single-JJ Ej in GHz from Lj in nH."""
    return (Phi0/(2*np.pi))**2 / (Lj_nH*1e-9) / (h_planck*1e9)

def f01_approx(Ej_eff_GHz, Ec_GHz):
    """f_01 = sqrt(8*Ej*Ec) - Ec  [transmon approximation]."""
    return np.sqrt(8*Ej_eff_GHz*Ec_GHz) - Ec_GHz

Ec = 0.140   # GHz — target Ec
print(f"Analytical preview  |  Ec = {Ec*1e3:.0f} MHz")
print(f"{'Lj/JJ (nH)':>12} {'Ej_eff (GHz)':>14} {'Ej/Ec':>8} {'f_01 (GHz)':>12}")
print("-" * 52)
for Lj_nH in [8, 9, 10, 11, 12, 13, 14, 15]:
    Ej      = Lj_to_Ej_GHz(Lj_nH)
    Ej_eff  = 2 * Ej          # two JJs in parallel: Ej_eff = 2*Ej_single
    ratio   = Ej_eff / Ec
    f01     = f01_approx(Ej_eff, Ec)
    tag     = "  <-- target" if 3.0 <= f01 <= 4.8 else ""
    print(f"{Lj_nH:>12.0f} {Ej_eff:>14.3f} {ratio:>8.0f} {f01:>12.4f}{tag}")


Analytical preview  |  Ec = 140 MHz
  Lj/JJ (nH)   Ej_eff (GHz)    Ej/Ec   f_01 (GHz)
----------------------------------------------------
           8         40.872      292       6.6259
           9         36.331      260       6.2389
          10         32.698      234       5.9116
          11         29.725      212       5.6300
          12         27.248      195       5.3843
          13         25.152      180       5.1676
          14         23.356      167       4.9745
          15         21.799      156       4.8011


---
## Step 3 — Render to HFSS and Configure Josephson Junctions

### What `sim.run()` does in HFSS
1. Creates a new eigenmode design named `Qbit_hfss`
2. Exports `metal_plane` (W×W with hole) → HFSS PEC sheet on substrate top face
3. Exports `disk` → HFSS PEC sheet (the qubit island)
4. Creates silicon substrate box (εr = 11.7) sized to the chip
5. Exports `jj_east` / `jj_west` → HFSS lumped RLC elements
6. Assigns `Lj` and `Cj` as HFSS design variables on each lumped element

### JJ HFSS object naming convention
Qiskit-Metal's HFSS renderer auto-generates names from the component name and
the key passed to `add_qgeometry('junction', {key: geom})`:
```
rect : JJ_rect_Lj_{component_name}_{key}
line : JJ_Lj_{component_name}_{key}_
```
For component `CCT1` with keys `jj_east` and `jj_west`:
```
jj_east rect : JJ_rect_Lj_CCT1_jj_east
jj_east line : JJ_Lj_CCT1_jj_east_
jj_west rect : JJ_rect_Lj_CCT1_jj_west
jj_west line : JJ_Lj_CCT1_jj_west_
```


In [8]:
from qiskit_metal.analyses.quantization import EPRanalysis

# ── Create EPRanalysis object ─────────────────────────────────────────────
eig_qb = EPRanalysis(design, "hfss")

# ── Simulation setup ──────────────────────────────────────────────────────
eig_qb.sim.setup.name          = 'Qbit_Setup'
eig_qb.sim.setup.n_modes       = 1
eig_qb.sim.setup.max_passes    = 15
eig_qb.sim.setup.max_delta_f   = 0.1    # convergence: 0.1% change in freq
eig_qb.sim.setup.min_freq_ghz  = 1.0
eig_qb.sim.setup.min_converged = 2

# ── Junction variables ────────────────────────────────────────────────────
# Lj=11 nH per JJ; Ej_eff = 2*Ej_single ~ 23.2 GHz
# At Ec=140 MHz: f_01 ~ 3.5 GHz, Ej/Ec ~ 166
eig_qb.sim.setup.vars = Dict(
    Lj = '11 nH',   # inductance per JJ — adjust to tune f_01
    Cj = '2 fF',    # shunt capacitance per JJ
)

# ── HFSS box buffer (airbox around chip) ──────────────────────────────────
eig_qb.sim.renderer.options['x_buffer_width_mm'] = 0.5
eig_qb.sim.renderer.options['y_buffer_width_mm'] = 0.5

print("EPRanalysis setup:")
eig_qb.sim.setup


EPRanalysis setup:


{'name': 'Qbit_Setup',
 'reuse_selected_design': True,
 'reuse_setup': True,
 'min_freq_ghz': 1.0,
 'n_modes': 1,
 'max_delta_f': 0.1,
 'max_passes': 15,
 'min_passes': 1,
 'min_converged': 2,
 'pct_refinement': 30,
 'basis_order': 1,
 'vars': {'Lj': '11 nH', 'Cj': '2 fF'}}

In [9]:
# ── Render design to HFSS ─────────────────────────────────────────────────
# Prerequisite: Ansys HFSS must be running with a valid licence.

eig_qb.sim.run(
    name='Qbit_hfss',
    components=['TC1'],
    open_terminations=[],   # no wave ports needed for eigenmode
    box_plus_buffer=True,   # auto-size HFSS airbox = chip + buffer
)

eig_qb.sim.print_run_args()
print()
print("HFSS rendering complete.")
print()

'''
# ── Print expected HFSS JJ object names (for EPR setup in Step 6) ─────────
comp = 'TC_1_1_n'
print("Expected HFSS JJ object names:")
for key in ['jj_east', 'jj_west']:
    print(f"  {key}:")
    print(f"    rect : JJ_rect_Lj_{comp}_{key}")
    print(f"    line : JJ_Lj_{comp}_{key}_")
'''


INFO 10:36PM [connect_project]: Connecting to Ansys Desktop API...
INFO 10:36PM [load_ansys_project]: 	Opened Ansys App
INFO 10:36PM [load_ansys_project]: 	Opened Ansys Desktop v2024.2.0
INFO 10:36PM [load_ansys_project]: 	Opened Ansys Project
	Folder:    C:/Users/Kobi/Documents/Ansoft/
	Project:   Project20
INFO 10:36PM [connect_design]: No active design found (or error getting active design).
INFO 10:36PM [connect]: 	 Connected to project "Project20". No design detected
INFO 10:37PM [connect_design]: 	Opened active design
	Design:    Qbit_hfss_hfss [Solution type: Eigenmode]
WARNING 10:37PM [connect_setup]: 	No design setup detected.
WARNING 10:37PM [connect_setup]: 	Creating eigenmode default setup.
INFO 10:37PM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.HfssEMSetup'>)
INFO 10:37PM [get_setup]: 	Opened setup `Qbit_Setup`  (<class 'pyEPR.ansys.HfssEMSetup'>)
INFO 10:37PM [analyze]: Analyzing setup Qbit_Setup
10:39PM 35s INFO [get_f_convergence]: Saved convergences to C:

This analysis object run with the following kwargs:
{'name': 'Qbit_hfss', 'components': ['TC1'], 'open_terminations': [], 'port_list': None, 'jj_to_port': None, 'ignored_jjs': None, 'box_plus_buffer': True}


HFSS rendering complete.



'\n# ── Print expected HFSS JJ object names (for EPR setup in Step 6) ─────────\ncomp = \'TC_1_1_n\'\nprint("Expected HFSS JJ object names:")\nfor key in [\'jj_east\', \'jj_west\']:\n    print(f"  {key}:")\n    print(f"    rect : JJ_rect_Lj_{comp}_{key}")\n    print(f"    line : JJ_Lj_{comp}_{key}_")\n'

---
## Step 4 — Run Finite-Element Eigenmode Analysis in HFSS

HFSS replaces each Josephson Junction with its linearised inductance Lj and
solves Maxwell's eigenvalue problem for the closed cavity.  The resulting
eigenfrequency is the **bare (linearised) qubit frequency** — slightly higher
than the true dressed frequency because the cosine nonlinearity is not yet
included (that is corrected in Step 9 by pyEPR quantum analysis).

Convergence criterion: < 0.1% change in eigenfrequency between consecutive
adaptive mesh refinement passes.


In [10]:
# ── The HFSS solve was triggered by sim.run() in Step 3. ──────────────────
# Use this cell to check convergence and re-run if needed.

# Re-run after changing setup parameters:
# eig_qb.sim.renderer.analyze_setup(eig_qb.sim.setup.name)

# ── Plot convergence ──────────────────────────────────────────────────────
eig_qb.sim.plot_convergences()
plt.suptitle(
    "HFSS Eigenmode Convergence — Grounded Concentric Transmon",
    fontsize=11, y=1.02
)
plt.tight_layout()
plt.savefig('convergence.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Extract eigenfrequencies ──────────────────────────────────────────────
freqs = eig_qb.get_frequencies()
print("Eigenmode frequencies (linearised with Lj):")
print(freqs)
print()
print("Convergence per pass:")
print(eig_qb.sim.convergence_f)


Design "Qbit_hfss_hfss" info:
	# eigenmodes    1
	# variations    1
Design "Qbit_hfss_hfss" info:
	# eigenmodes    1
	# variations    1
Eigenmode frequencies (linearised with Lj):
                Freq. (GHz)  Quality Factor
variation mode                             
0         0        4.335365             inf

Convergence per pass:
         re(Mode(1)) [g]
Pass []                 
1               3.465145
2               4.074097
3               4.233014
4               4.285400
5               4.304045
6               4.316294
7               4.323573
8               4.329389
9               4.332358
10              4.335365
11                   NaN
12                   NaN
13                   NaN
14                   NaN
15                   NaN


---
## Step 5 — Plot Electric and Magnetic Field Distributions

Expected field pattern for the **grounded concentric transmon**:

| Field | Expected distribution |
|---|---|
| **E-field** | Radially concentrated across the circular slot (Wslot gap). Strong radial E in the slot, falls off quickly inside the disk and in the metallic plane. |
| **H-field** | Circulates around the disk; peaks at the JJ bridges (east/west) where current loops close through the JJ inductance. |

A large fraction of E-field inside the substrate is expected (~90%),
consistent with typical planar transmon designs on silicon.


In [11]:
# ── E-field magnitude on chip surface ────────────────────────────────────
print("Plotting E-field (eigenmode 1)...")
eig_qb.sim.plot_fields('main')
eig_qb.sim.save_screenshot('efield_main.png')
print("  Saved: efield_main.png")

# ── H-field ──────────────────────────────────────────────────────────────
print("Plotting H-field (eigenmode 1)...")
eig_qb.sim.plot_fields('main', field_type='H')
eig_qb.sim.save_screenshot('hfield_main.png')
print("  Saved: hfield_main.png")

eig_qb.sim.clear_fields()

print()
print("Interpretation checklist:")
print("  [ ] E-field peaks inside the circular slot (Wslot gap region)")
print("  [ ] E-field drops sharply outside slot and inside disk")
print("  [ ] H-field circulates around disk; peaks at +/-x JJ positions")
print("  [ ] No significant field at chip boundary (W x W square edge)")
print()
print("If E-field leaks to the chip boundary, increase W to reduce")
print("boundary coupling and improve accuracy.")


INFO 10:39PM [get_setup]: 	Opened setup `Qbit_Setup`  (<class 'pyEPR.ansys.HfssEMSetup'>)


Plotting E-field (eigenmode 1)...


TypeError: QSimulation.save_screenshot() takes 1 positional argument but 2 were given

---
## Step 6 — EPR Analysis on the Qubit Mode

pyEPR integrates the stored electric energy over each JJ rectangle to
compute the inductive Energy Participation Ratio (EPR):

```
p_mj = (inductive energy in JJ j for mode m) / (total electric energy)
```

For the grounded concentric transmon with two symmetric JJs:
- `p_total = p_east + p_west`
- Ideally `p_total → 1` (all inductive energy in the JJs)
- Typical values: 0.85 – 0.98 for well-designed transmons

The capacitive EPR `p_cap` (small for transmon) and sign `s_mj` are also reported.


In [12]:
# ── Configure junction rectangles and integration lines ───────────────────
comp = 'CCT1'

# Remove default single-junction placeholder if present
if 'jj' in eig_qb.setup.junctions:
    del eig_qb.setup.junctions['jj']

# Register east JJ
eig_qb.setup.junctions['jj_east'] = Dict(
    rect        = f'JJ_rect_Lj_{comp}_jj_east',
    line        = f'JJ_Lj_{comp}_jj_east_',
    Lj_variable = 'Lj',
    Cj_variable = 'Cj',
)

# Register west JJ
eig_qb.setup.junctions['jj_west'] = Dict(
    rect        = f'JJ_rect_Lj_{comp}_jj_west',
    line        = f'JJ_Lj_{comp}_jj_west_',
    Lj_variable = 'Lj',
    Cj_variable = 'Cj',
)

# Substrate dielectric name (must match the HFSS 3D object name)
eig_qb.setup.dissipatives.dielectrics_bulk = ['main']

print("EPR junction configuration:")
print(eig_qb.setup)
print(eig_qb.setup.junctions)


EPR junction configuration:
{'junctions': {'jj_east': {'rect': 'JJ_rect_Lj_CCT1_jj_east', 'line': 'JJ_Lj_CCT1_jj_east_', 'Lj_variable': 'Lj', 'Cj_variable': 'Cj'}, 'jj_west': {'rect': 'JJ_rect_Lj_CCT1_jj_west', 'line': 'JJ_Lj_CCT1_jj_west_', 'Lj_variable': 'Lj', 'Cj_variable': 'Cj'}}, 'dissipatives': {'dielectrics_bulk': ['main']}, 'cos_trunc': 8, 'fock_trunc': 7, 'sweep_variable': 'Lj'}
{'jj_east': {'rect': 'JJ_rect_Lj_CCT1_jj_east', 'line': 'JJ_Lj_CCT1_jj_east_', 'Lj_variable': 'Lj', 'Cj_variable': 'Cj'}, 'jj_west': {'rect': 'JJ_rect_Lj_CCT1_jj_west', 'line': 'JJ_Lj_CCT1_jj_west_', 'Lj_variable': 'Lj', 'Cj_variable': 'Cj'}}


In [13]:
# ── Run EPR analysis ──────────────────────────────────────────────────────
print("Running EPR field integration...")
eig_qb.run_epr()

print()
print("Raw EPR results:")
print(eig_qb.epr_results)


Running EPR field integration...


AssertionError: pyEPR ProjectInfo user error found 😷:
                    Seems like for junction `jj_east` you specified a rect that does not exist
                    in HFSS by the name: `JJ_rect_Lj_CCT1_jj_east` 

In [14]:
#eig_qb.sim.close()

---
## Step 7 — Qubit Frequency and Anharmonicity

From first-order EPR perturbation theory:

| Quantity | Formula |
|---|---|
| Corrected f_01 | `f_q = f_HFSS * (1 - p_total / 2)` |
| Anharmonicity | `alpha = - p_total^2 * Ec` |
| Ec estimate | invert `f_01 = sqrt(8*Ej_eff*Ec) - Ec` |
| Ej_eff | `= 2 * (Phi0/2pi)^2 / Lj`  (two JJs in parallel) |

The precise Ec and dressed frequency are calculated in Step 9.


In [15]:
h_planck = 6.626e-34
Phi0     = 2.068e-15

# ── HFSS bare eigenfrequency ──────────────────────────────────────────────
f_hfss_GHz = float(eig_qb.get_frequencies().iloc[0, 0])

# ── Inductive EPR per junction ────────────────────────────────────────────
res    = eig_qb.epr_results
# pyEPR may store these in different dict structures depending on version:
def get_epr_p(key):
    for path in [res.get('p_mj', {}), res]:
        v = path.get(key)
        if v is not None:
            return float(v)
    return 0.0

p_east  = get_epr_p('jj_east')
p_west  = get_epr_p('jj_west')
p_total = p_east + p_west

print("=" * 58)
print(f"  HFSS bare eigenfrequency  : {f_hfss_GHz:.4f} GHz")
print(f"  p_inductive jj_east       : {p_east:.4f}")
print(f"  p_inductive jj_west       : {p_west:.4f}")
print(f"  p_total (east + west)     : {p_total:.4f}")
print("=" * 58)

# First-order corrected qubit frequency
f_q_GHz = f_hfss_GHz * (1.0 - p_total / 2.0)
print(f"\n  Corrected f_01 (EPR O1)   : {f_q_GHz:.4f} GHz")

# ── Ej from Lj ───────────────────────────────────────────────────────────
Lj_nH       = float(eig_qb.sim.setup.vars.Lj.replace(' nH', '').strip())
Ej_single   = (Phi0/(2*np.pi))**2 / (Lj_nH*1e-9) / (h_planck*1e9)
Ej_eff_GHz  = 2 * Ej_single

print(f"\n  Lj (per JJ)               : {Lj_nH:.1f} nH")
print(f"  Ej per JJ                 : {Ej_single:.3f} GHz")
print(f"  Ej_eff (two JJs parallel) : {Ej_eff_GHz:.3f} GHz")

# ── Back-estimate Ec ──────────────────────────────────────────────────────
def f01_eq(Ec, Ej, target):
    return np.sqrt(8*Ej*Ec) - Ec - target

try:
    Ec_GHz     = brentq(f01_eq, 0.01, 1.0, args=(Ej_eff_GHz, f_hfss_GHz))
    alpha_GHz  = -(p_total**2) * Ec_GHz
    EjEc_ratio = Ej_eff_GHz / Ec_GHz

    print(f"\n  Estimated Ec              : {Ec_GHz*1e3:.1f} MHz")
    print(f"  Estimated anharmonicity   : {alpha_GHz*1e3:.1f} MHz")
    print(f"  Estimated Ej/Ec           : {EjEc_ratio:.1f}")

    ok_f  = "PASS" if 3.0 <= f_hfss_GHz <= 4.8 else "FAIL -- adjust Lj"
    ok_ej = "PASS" if EjEc_ratio > 50 else "FAIL -- reduce Lj"
    print(f"\n  Target: f_01 in 3-4.8 GHz : {f_hfss_GHz:.4f} GHz  [{ok_f}]")
    print(f"  Target: Ej/Ec > 50        : {EjEc_ratio:.1f}       [{ok_ej}]")
except Exception as ex:
    print(f"  Could not estimate Ec: {ex}")
    Ec_GHz     = 0.140
    EjEc_ratio = Ej_eff_GHz / Ec_GHz


com_error: (-2147023174, 'The RPC server is unavailable.', None, None)

---
## Step 8 — Substrate EPR and Surface Participation Ratio (SPR)

### 8a  Bulk substrate dielectric participation
The fraction of the total electric energy stored inside the bulk silicon
substrate.  Computed automatically during `run_epr()` and printed as
`EPR of substrate = XX.X%`.

**T1 budget (dielectric)**:
```
1/T1_diel = p_sub * omega_01 * tan(delta_sub)
```

### 8b  Surface Participation Ratio (SPR)
Energy density at nm-scale interfacial loss layers:

| Interface | Notation | t (nm) | eps_r | tan_delta |
|---|---|---|---|---|
| Metal–Air | MA | 3 | 10 | ~1e-3 |
| Metal–Substrate | MS | 2 | 11.4 | ~1e-3 |
| Substrate–Air | SA | 2 | 11.4 | ~1e-3 |

```
p_surf = (eps_r * eps_0 * t) / (2 * U_E) * Integral|E_parallel|^2 dA
```
Requires 2D Sheet objects in HFSS representing each interface.


In [16]:
# ── 8a: Bulk substrate EPR ────────────────────────────────────────────────
print("=== 8a: Bulk Substrate EPR ===")
print()

# pyEPR prints 'EPR of substrate = XX%' during run_epr().
# The value is also stored as p_dielectric_main_0 in epr_results.
try:
    p_sub = float(eig_qb.epr_results['p_dielectric_main_0'])
    print(f"  p_substrate (bulk Si)     : {p_sub:.4f}  ({p_sub*100:.1f}%)")
except (KeyError, TypeError):
    p_sub = 0.92   # typical value for planar transmon on Si
    print(f"  Using estimate            : p_sub ~ {p_sub} (read from run_epr output)")

# Dielectric-limited T1
tan_delta_Si = 3e-6          # bulk Si at mK (high-purity)
omega_q      = 2 * np.pi * f_hfss_GHz * 1e9
Q_diel       = 1.0 / (p_sub * tan_delta_Si)
T1_diel_us   = Q_diel / omega_q * 1e6

print(f"  tan(delta) Si assumed     : {tan_delta_Si:.0e}")
print(f"  Dielectric Q              : {Q_diel:.2e}")
print(f"  Dielectric-limited T1     : {T1_diel_us:.0f} us")


=== 8a: Bulk Substrate EPR ===



AttributeError: 'EPRanalysis' object has no attribute 'epr_results'

In [17]:
# ── 8b: Surface Participation Ratio ──────────────────────────────────────
print("=== 8b: Surface Participation Ratio (SPR) ===")
print()

eps_0 = 8.854e-12   # F/m

# Interface parameters
surfaces = {
    'MA': {'eps_r': 10.0,  't_m': 3e-9,  'tan_d': 1e-3,  'label': 'metal-air'},
    'MS': {'eps_r': 11.4,  't_m': 2e-9,  'tan_d': 1e-3,  'label': 'metal-substrate'},
    'SA': {'eps_r': 11.4,  't_m': 2e-9,  'tan_d': 1e-3,  'label': 'substrate-air'},
}

print("  Full SPR calculation requires 2D Sheet objects in HFSS.")
print("  Create sheets at each interface, then use the field calculator.")
print()
print("  Template (pyEPR + HFSS COM interface):")
print("""    renderer = eig_qb.sim.renderer
    pinfo    = renderer.pinfo
    U_E      = eig_qb.epr_results['U_E']   # total electric energy [J]

    sheet_names = {
        'MA': 'MA_surface',   # HFSS Sheet object name for metal-air interface
        'MS': 'MS_surface',   # HFSS Sheet object name for metal-substrate
        'SA': 'SA_surface',   # HFSS Sheet object name for substrate-air
    }

    for iface, sp in surfaces.items():
        # Field integral via HFSS calculator:
        # E_sq = pinfo.get_surface_tangential_Esquared(sheet_names[iface])
        # p_surf = sp['eps_r'] * eps_0 * sp['t_m'] * E_sq / (2 * U_E)
        # Q_surf = 1.0 / (p_surf * sp['tan_d'])
        # T1_surf_us = Q_surf / omega_q * 1e6
        pass
""")

# ── Proxy estimate ────────────────────────────────────────────────────────
print("  Proxy estimate (order of magnitude):")
t_sub_m  = 500e-6    # 500 um Si wafer
p_MA_est = p_sub * (surfaces['MA']['eps_r'] * surfaces['MA']['t_m']) /                     (11.4 * t_sub_m)
T1_MA_us = (1.0 / (p_MA_est * surfaces['MA']['tan_d'])) / omega_q * 1e6

print(f"    p_MA (metal-air proxy)   : {p_MA_est:.2e}")
print(f"    T1_MA (tan_d=1e-3)       : {T1_MA_us:.0f} us")
print()
print("  Dominant loss channel for this geometry: bulk substrate (p_sub ~ 90%)")
print("  MA surface loss: secondary; reduce by increasing W or Rdisk.")


=== 8b: Surface Participation Ratio (SPR) ===

  Full SPR calculation requires 2D Sheet objects in HFSS.
  Create sheets at each interface, then use the field calculator.

  Template (pyEPR + HFSS COM interface):
    renderer = eig_qb.sim.renderer
    pinfo    = renderer.pinfo
    U_E      = eig_qb.epr_results['U_E']   # total electric energy [J]

    sheet_names = {
        'MA': 'MA_surface',   # HFSS Sheet object name for metal-air interface
        'MS': 'MS_surface',   # HFSS Sheet object name for metal-substrate
        'SA': 'SA_surface',   # HFSS Sheet object name for substrate-air
    }

    for iface, sp in surfaces.items():
        # Field integral via HFSS calculator:
        # E_sq = pinfo.get_surface_tangential_Esquared(sheet_names[iface])
        # p_surf = sp['eps_r'] * eps_0 * sp['t_m'] * E_sq / (2 * U_E)
        # Q_surf = 1.0 / (p_surf * sp['tan_d'])
        # T1_surf_us = Q_surf / omega_q * 1e6
        pass

  Proxy estimate (order of magnitude):


NameError: name 'p_sub' is not defined

---
## Step 9 — Quantum Analysis: Ej, Ec, Ej/Ec and PSR

`analyze_all_variations()` reconstructs the full Josephson Hamiltonian
from the EPR participation ratios, using:
- Non-degenerate perturbation theory (ND): most accurate
- First-order perturbation (O1): faster cross-check

**Key outputs**:
| Symbol | Source in results dict | Units |
|---|---|---|
| f_01 | `results['f_ND'][0]` | GHz |
| Ec (= -alpha) | `-results['chi_ND'][0,0]` | GHz |
| alpha | `results['chi_ND'][0,0]` | GHz |
| Ej per JJ | `get_Ejs()` | GHz |
| Ej/Ec | `Ej_eff / Ec` | — |
| PSR (sign) | `epr_results['sign_mj']` | ±1 |


In [ ]:
# ── Run full quantum analysis ─────────────────────────────────────────────
print("Running analyze_all_variations()...")
print("  cos_trunc = 8  — Taylor expansion of cos(phi) to 8th order")
print("  fock_trunc = 7 — Fock space size: levels |0> to |6>")
print()

eig_qb.analyze_all_variations(
    cos_trunc  = 8,
    fock_trunc = 7,
)

print("Quantum analysis complete.")


In [ ]:
# ── Extract Ej, Ec, Ej/Ec ────────────────────────────────────────────────
variation = '0'

print("=" * 62)
print("  QUANTUM HAMILTONIAN PARAMETERS")
print("=" * 62)

# ── Josephson energies ────────────────────────────────────────────────────
try:
    Ejs        = eig_qb.get_Ejs(variation=variation)
    print("\n  Josephson energies (GHz):")
    for jname, jval in Ejs.items():
        print(f"    {jname:20s}: {float(jval):.4f} GHz  "
              f"({float(jval)*1e3:.1f} MHz)")
    Ej_eff_qa = sum(float(v) for v in Ejs.values())
    print(f"    {'Ej_eff (sum, 2 JJ ||)':20s}: {Ej_eff_qa:.4f} GHz")
except Exception as ex:
    print(f"  [Ejs: {ex}]")
    Ej_eff_qa = Ej_eff_GHz

# ── Chi matrix → Ec, anharmonicity ───────────────────────────────────────
try:
    res      = eig_qb.results[variation]
    f_ND     = res['f_ND']
    f_O1     = res['f_O1']
    chi_ND   = res['chi_ND']

    Ec_ND    = -float(chi_ND[0, 0])   # GHz
    alpha_ND =  float(chi_ND[0, 0])   # GHz (negative for transmon)
    EjEc     = Ej_eff_qa / Ec_ND

    print(f"\n  Dressed qubit frequency:")
    print(f"    f_ND  (pert. theory)  : {float(f_ND[0]):.6f} GHz")
    print(f"    f_O1  (1st order)     : {float(f_O1[0]):.6f} GHz")

    print(f"\n  Charging energy Ec    : {Ec_ND*1e3:.2f} MHz")
    print(f"  Anharmonicity alpha   : {alpha_ND*1e3:.2f} MHz")

    print(f"\n  ─── Ej/Ec ───")
    print(f"    Ej_eff              : {Ej_eff_qa:.4f} GHz = {Ej_eff_qa*1e3:.1f} MHz")
    print(f"    Ec                  : {Ec_ND*1e3:.2f} MHz")
    print(f"    Ej/Ec               : {EjEc:.1f}")

    ok_f  = "PASS" if 3.0 <= float(f_ND[0]) <= 4.8  else "FAIL"
    ok_ec = "PASS" if 120   <= Ec_ND*1e3    <= 160   else "FAIL"
    ok_ej = "PASS" if EjEc  > 50                     else "FAIL"

    print(f"\n  ─── Target check ───")
    print(f"    f_01 in 3–4.8 GHz   : {float(f_ND[0]):.4f} GHz  [{ok_f}]")
    print(f"    Ec  in 120–160 MHz  : {Ec_ND*1e3:.1f} MHz  [{ok_ec}]")
    print(f"    Ej/Ec > 50          : {EjEc:.1f}         [{ok_ej}]")

    if ok_f == "FAIL":
        if float(f_ND[0]) < 3.0:
            print("    --> Increase Lj (or decrease W/Rdisk) to raise f_01")
        else:
            print("    --> Decrease Lj (or increase W/Rdisk) to lower f_01")
    if ok_ec == "FAIL":
        print("    --> Adjust Rdisk or Wslot: larger disk/slot -> lower Ec")

except Exception as ex:
    print(f"  [chi_ND not available: {ex}]")


In [ ]:
# ── PSR: Participation Sign Ratio ────────────────────────────────────────
print("=" * 62)
print("  PARTICIPATION SIGN RATIO (PSR  /  s_mj)")
print("=" * 62)
print()

try:
    epr_data = eig_qb.epr_results
    # Check common key names for sign data
    for key in ['sign_mj', 'SOJ', 'sign', 'S']:
        if key in epr_data:
            print(f"  Signs from epr_results['{key}']:")
            print(epr_data[key])
            print()
            break
    else:
        print("  PSR is printed in the run_epr() output above.")
        print("  Look for lines:   'sign s_0j    (+)'")
        print()

    print("""
  Interpretation:
    s = +1 : JJ participates constructively — current flows in the
             positive sense through this junction for this mode.
    s = -1 : JJ participates with reversed polarity.

  Expected for CCT1 (symmetric layout):
    jj_east: s_0j = +1
    jj_west: s_0j = +1
  Both signs equal (+1) confirms the JJs participate in parallel and
  add coherently in the qubit mode, as designed.

  If one sign is -1: check the JJ line direction in the HFSS 3D model.
  The integration line must be drawn in a consistent radial direction
  for both junctions.
    """)

except Exception as ex:
    print(f"  [{ex}]")


In [ ]:
# ── Final design summary ──────────────────────────────────────────────────
print("=" * 65)
print("  DESIGN SUMMARY  —  Grounded Concentric Transmon")
print("=" * 65)

print(f"\n  Geometry")
print(f"    W         : {cct.options.W}  (metallic plane side)")
print(f"    Rdisk     : {cct.options.Rdisk}  (disk radius)")
print(f"    Wslot     : {cct.options.Wslot}  (slot width = Rdisk to Rdisk+Wslot)")
print(f"    jj_width  : {cct.options.jj_width}")

print(f"\n  HFSS simulation")
print(f"    Lj / JJ  : {eig_qb.sim.setup.vars.Lj}")
print(f"    Cj / JJ  : {eig_qb.sim.setup.vars.Cj}")
print(f"    Bare f   : {f_hfss_GHz:.4f} GHz")

try:
    print(f"\n  Quantum Hamiltonian")
    print(f"    f_01     : {float(f_ND[0]):.4f} GHz")
    print(f"    Ec       : {Ec_ND*1e3:.2f} MHz")
    print(f"    alpha    : {alpha_ND*1e3:.2f} MHz")
    print(f"    Ej_eff   : {Ej_eff_qa:.4f} GHz")
    print(f"    Ej/Ec    : {EjEc:.1f}")
    print(f"\n  Loss estimates")
    print(f"    p_sub    : {p_sub:.3f} ({p_sub*100:.1f}%)")
    print(f"    T1_diel  : {T1_diel_us:.0f} us")
    print(f"    p_MA est : {p_MA_est:.2e}")
    print(f"    T1_MA est: {T1_MA_us:.0f} us")
except NameError:
    print("    (Complete Steps 7-9 to populate)")
print("=" * 65)


---
## Tuning Guide

| Goal | Action | Effect |
|---|---|---|
| ↑ f_01 | Decrease `Lj` | ↑ Ej → ↑ f |
| ↓ f_01 | Increase `Lj` | ↓ Ej → ↓ f |
| ↑ Ec (more anharmonic) | Decrease `Rdisk` | ↓ CΣ → ↑ Ec |
| ↓ Ec | Increase `Rdisk` or `Wslot` | ↑ CΣ → ↓ Ec |
| Reduce substrate loss | Decrease `W` (smaller ground plane area) | Less substrate E-field |
| Reduce MA surface loss | Increase `Rdisk` + `W` | E-field more dilute over larger area |
| Stronger isolation | Increase `W` | More ground plane around the qubit |

**Lj convention**: Each JJ is assigned inductance `Lj` in HFSS.
The two JJs in parallel give `Lj_eff = Lj/2` and `Ej_eff = 2 * (Phi0/2pi)^2 / Lj`.


In [ ]:
# ── Optional: close HFSS ──────────────────────────────────────────────────
# eig_qb.sim.close()
# gui.main_window.close()
print("Notebook complete.")
print("Uncomment the two lines above to close HFSS and the Metal GUI.")
